In [ ]:
# 03_variance_validation.ipynb
# Validates the U-statistic mean and variance formulas for the contact count
# under the seeding/counting convention at r = 0.002, n = 600.
import os
import numpy as np
import pandas as pd

TAB_DIR = "../results/tables"
os.makedirs(TAB_DIR, exist_ok=True)

rng = np.random.default_rng(20261006)

ell = np.array([0.2, 0.4])
p = np.array([0.5, 0.5])
lmax = ell.max()
Lbar = float((p * ell).sum())
Cvol, Svol = 1.0, (1.0 + lmax) ** 3

# Closed-form truncated moments (needed for the variance prediction)
raw = lambda k: float((p * ell ** k).sum())
lam1 = raw(1) * Cvol / Svol
lam2 = (raw(2) - 0.5 * raw(3) + raw(4) / (3 * np.pi) - raw(5) / (40 * np.pi)) / Svol
var_lc = lam2 - lam1 ** 2

r, n, reps = 0.002, 600, 3500
twor = 2 * r

# --- Predicted mean and variance ---
q = np.pi * r * Lbar ** 2 * Cvol / Svol ** 2
EN_pred = n * (n - 1) / 2 * q
zeta1 = (np.pi * r * Lbar / Svol) ** 2 * var_lc
Var_pred = n * (n - 1) * (n - 2) * zeta1 + n * (n - 1) / 2 * q * (1 - q)


def seg_seg_dist_mid(P1, P2, Q1, Q2):
    d1, d2, rr = P2 - P1, Q2 - Q1, P1 - Q1
    a = (d1 * d1).sum(1); e = (d2 * d2).sum(1); f = (d2 * rr).sum(1)
    b = (d1 * d2).sum(1); c = (d1 * rr).sum(1)
    den = a * e - b * b
    s = np.where(den > 1e-12, np.clip((b * f - c * e) / den, 0, 1), 0.0)
    t = np.clip((b * s + f) / e, 0, 1)
    s = np.clip((b * t - c) / a, 0, 1)
    cpA, cpB = P1 + s[:, None] * d1, Q1 + t[:, None] * d2
    return np.linalg.norm(cpA - cpB, axis=1), 0.5 * (cpA + cpB)


def count_once(n, twor, rng):
    L = rng.choice(ell, size=n, p=p)
    X = rng.uniform(-lmax / 2, 1 + lmax / 2, size=(n, 3))
    v = rng.normal(size=(n, 3)); v /= np.linalg.norm(v, axis=1, keepdims=True)
    P1, P2 = X - 0.5 * L[:, None] * v, X + 0.5 * L[:, None] * v
    i, j = np.triu_indices(n, 1)
    dist, mid = seg_seg_dist_mid(P1[i], P2[i], P1[j], P2[j])
    inC = np.all((mid >= 0) & (mid <= 1), axis=1)
    return int(np.sum((dist <= twor) & inC))


counts = np.array([count_once(n, twor, rng) for _ in range(reps)])
EN_obs, Var_obs = counts.mean(), counts.var(ddof=1)

table = pd.DataFrame({
    "quantity": ["E[N]", "Var(N)"],
    "predicted": [EN_pred, Var_pred],
    "observed": [EN_obs, Var_obs],
    "ratio": [EN_obs / EN_pred, Var_obs / Var_pred],
})
print(f"r={r}, n={n}, reps={reps}")
print(table.to_string(index=False))
table.to_latex(os.path.join(TAB_DIR, "table_variance.tex"), index=False, float_format="%.3f")
